##### Imports

In [22]:
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from econml.dml import CausalForestDML

from gg570_d200.auxiliary_functions.forest_riesz_funcs import call_forestriesz, call_forestriesz_cross
from gg570_d200.auxiliary_functions.ate_estimation_funcs import forest_riesz_gate, causal_dml_gate

In [23]:
root = Path.cwd().parent
data_path = root / "data"

In [24]:
np.random.seed(21)

In [25]:
df_scaled = pd.read_csv(data_path / "df_scaled.csv")

In [26]:
saved_joblib = joblib.load(data_path / "covariate_scaler.joblib")
scaler = saved_joblib['scaler']

In [27]:
df = pd.read_csv(data_path / "df.csv")

In [28]:
treatment_col = 'Training (last 3 months)'
outcome_col = 'Underemployment hours'
covariate_cols = [col for col in df_scaled.columns if col not in treatment_col and col not in outcome_col and col not in ['prop_scores']]

---

##### Key specification

In [29]:
cross_scale = call_forestriesz_cross(df_scaled, covariate_cols, treatment_col, outcome_col, methods=['dr', 'plugin'])
cross_scale

{'dr': {'est': -0.16881398495582642,
  'low': -1.3646790706040013,
  'high': 1.0270511006923482,
  'p_val': 0.7820275135261567},
 'plugin': {'est': -0.03872457663475174,
  'low': -0.4595074374118259,
  'high': 0.3820582841423225,
  'p_val': 0.8568580545187574}}

---

##### Alternative specifications

###### ForestRiesz alternatives

In [30]:
no_cross_scale, est = call_forestriesz(df_scaled, covariate_cols, treatment_col, outcome_col, methods=['dr', 'plugin'], return_est=True)
no_cross_scale

{'dr': {'est': -0.2547221541217664,
  'low': -0.8629824111996713,
  'high': 0.35353810295613847,
  'p_val': 0.4117730965626576},
 'plugin': {'est': -0.0367478621002449,
  'low': -0.296532478464972,
  'high': 0.22303675426448222,
  'p_val': 0.7815905097974638}}

In [31]:
heterogeneity_importance = est.feature_importances_

In [32]:
top_3_ids = np.argsort(heterogeneity_importance)[-3:][::-1]
for id in top_3_ids:
    print(f"{covariate_cols[id]}: {heterogeneity_importance[id]}")

LKWFWM_bin_(12.2, 15.0]: 0.4264092363788278
FTPTWK_bin_(1.5, 2.0]: 0.0867071098779133
FTPTWK_bin_(0.999, 1.5]: 0.07434881656949945


In [34]:
mask = df_scaled[f'LKWFWM_bin_(12.2, 15.0]'] > 0.0
print(forest_riesz_gate(df_scaled, covariate_cols, treatment_col, outcome_col, est, mask))
print(forest_riesz_gate(df_scaled, covariate_cols, treatment_col, outcome_col, est, ~mask))

[-0.06610712231675214, -0.7141819729454322, 0.5819677283119278, 0.8415377724871507]
[-0.9999364464494207, -2.589237235223906, 0.5893643423250643, 0.21752141262800162]


In [35]:
mask = df_scaled[f'FTPTWK_bin_(1.5, 2.0]'] > 0.0
print(forest_riesz_gate(df_scaled, covariate_cols, treatment_col, outcome_col, est, mask))
print(forest_riesz_gate(df_scaled, covariate_cols, treatment_col, outcome_col, est, ~mask))

[-0.9734022195622224, -1.7571872671471382, -0.18961717197730665, 0.01492770664809484]
[0.5403972926272675, -0.401702981432283, 1.482497566686818, 0.2609056338019693]


---

In [13]:
no_cross_no_scale, est = call_forestriesz(df, covariate_cols, treatment_col, outcome_col, methods=['dr', 'plugin'], return_est=True)
no_cross_no_scale

{'dr': {'est': -0.2563584130844099,
  'low': -0.8648298709317971,
  'high': 0.35211304476297733,
  'p_val': 0.4089385090765416},
 'plugin': {'est': -0.037142018151062196,
  'low': -0.2968813883836602,
  'high': 0.22259735208153578,
  'p_val': 0.7792707652518849}}

In [14]:
cross_no_scale = call_forestriesz_cross(df, covariate_cols, treatment_col, outcome_col, methods=['dr', 'plugin'])
cross_no_scale

{'dr': {'est': -0.16996809045336825,
  'low': -1.366777810083686,
  'high': 1.0268416291769495,
  'p_val': 0.7807441141944769},
 'plugin': {'est': -0.03936654703911537,
  'low': -0.460119042457307,
  'high': 0.3813859483790763,
  'p_val': 0.8545009456647903}}

---

###### CausalForestDML

In [15]:
causal_dml = CausalForestDML(
    discrete_treatment=True,
    criterion='mse',
    n_estimators=100,
    min_samples_leaf=2,
    min_var_fraction_leaf=0.001,
    min_var_leaf_on_val=True,
    min_impurity_decrease=0.01,
    #max_samples=.8,
    max_depth=None,
    inference=True, #inference=False
    subforest_size=2, #subforest_size=1
    honest=True,
    verbose=0,
    n_jobs=-2,
    random_state=21,
    cv=3
)

In [16]:
causal_dml.fit(df_scaled[outcome_col], df_scaled[treatment_col], X=df_scaled[covariate_cols])
causal_dml.ate_, causal_dml.ate_stderr_

(array([-0.03379913]), array([0.42109106]))

In [17]:
heterogeneity_importance = causal_dml.feature_importances_

In [18]:
top_3_ids = np.argsort(heterogeneity_importance)[-3:][::-1]
for id in top_3_ids:
    print(f"{covariate_cols[id]}: {heterogeneity_importance[id]}")

AGE: 0.12016064204929525
REFWKD: 0.06559192860626194
HIQUAL22: 0.05090186012900608


In [37]:
#### cont here

In [19]:
aage_bins = ["(1.99, 4.0]", "(10.0, 12.0]", "(4.0, 6.0]", "(6.0, 8.0]", "(8.0, 10.0]"]

In [20]:
for bin in aage_bins:
    mask = df_scaled[f'AAGE_bin_{bin}'] > 0.0
    print(causal_dml_gate(df, covariate_cols, causal_dml, mask))
    mask = df_scaled[f'AAGE_bin_{bin}'] < 0.0
    print(causal_dml_gate(df, covariate_cols, causal_dml, mask),"\n")

[0.4439220481437849, -4.238005028477581, 5.12584912476515, 0.8525732674546236]
[0.43759330543594094, -5.03559592906841, 5.910782539940291, 0.875478642420662] 

[0.897982680518652, -7.8899359271614244, 9.685901288198727, 0.8412643287892896]
[0.3574716356412667, -4.230781133585722, 4.945724404868256, 0.8786339440360031] 

[0.26793513972352884, -4.185432863851554, 4.721303143298612, 0.9061306779659222]
[0.48565221593514063, -5.180829157094368, 6.152133588964648, 0.8665982758016404] 

[0.32440580400606295, -4.18400889770629, 4.8328205057184155, 0.8878458979772033]
[0.48432692206265154, -5.271650660785073, 6.240304504910374, 0.8690086594137885] 

[0.4423001077324799, -4.308591924174136, 5.193192139639096, 0.8552143890995381]
[0.4363769277624678, -5.223824119385391, 6.096577974910326, 0.8798931137315333] 

